# Uncertainty Estimation Practical — Part I
This practical studies **deep ensembles** for predictive uncertainty in regression and classification, based on [Lakshminarayanan, Pritzel and Blundell (2017)](https://arxiv.org/abs/1612.01474). Part II studies **multi-input multi-output (MIMO)** networks.

**Tasks.** Complete the two NLL derivations, the Gaussian and Laplace NLL functions, their adversarial-training variants, the regression mixture variance, and the classification adversarial-training function. Then answer the discussion questions using your results. A `NotImplementedError` marks an intentional student task, not a working default.

**Running in Google Colab.** Open the `.ipynb` using **File > Upload notebook**, connect to a Python runtime, and run cells in order. A CPU runtime is sufficient for the core exercises. Use the runtime's existing `jax`, `numpy`, and `matplotlib`; no toy-data download is needed. If an import reports a missing package, run `%pip install jax numpy matplotlib` in a setup cell before the imports, then restart the runtime. For local Jupyter, follow `README.md`. After changing a training setting, restart the runtime/kernel and run all cells again: JAX-compiled functions may otherwise retain previously captured optimizer settings. Training settings remain at the start of each training cell.

**Note.** for this year an LLM has been used to polish this course exercise. Please report to the teachers if you encounter an issue with the notebook.


## Simple and Scalable Predictive Uncertainty Estimation Using Deep Ensembles
**Balaji Lakshminarayanan, Alexander Pritzel and Charles Blundell (2017)**

### Original abstract (2017)
The following is the paper's original abstract; its statements about what is “current” are historical, not a comparison of methods today.

Deep neural networks (NNs) are powerful black box predictors that have recently achieved impressive performance on a wide spectrum of tasks. Quantifying predictive uncertainty in NNs is a challenging and yet unsolved problem. Bayesian NNs, which learn a distribution over weights, are currently the state-of-the-art for estimating predictive uncertainty; however these require significant modifications to the training procedure and are computationally expensive compared to standard (non-Bayesian) NNs. We propose an alternative to Bayesian NNs that is simple to implement, readily parallelizable, requires very little hyperparameter tuning, and yields high quality predictive uncertainty estimates. Through a series of experiments on classification and regression benchmarks, we demonstrate that our method produces well-calibrated uncertainty estimates which are as good or better than approximate Bayesian NNs. To assess robustness to dataset shift, we evaluate the predictive uncertainty on test examples from known and unknown distributions, and show that our method is able to express higher uncertainty on out-of-distribution examples. We demonstrate the scalability of our method by evaluating predictive uncertainty estimates on ImageNet.

### Notation
Let the training observations be $(\boldsymbol{x}_n,y_n)_{n=1}^N$, with $\boldsymbol{x}_n\in\mathbb{R}^D$. For classification, $y_n\in\{0,\ldots,K-1\}$, matching the code; for regression, $y_n\in\mathbb{R}$. A network with parameters $\theta$ models $p_\theta(y\mid\boldsymbol{x})$.

### Proper scoring rules
A scoring rule $S(p,(y,\boldsymbol{x}))$ evaluates a predictive distribution using an observed outcome. Under the **higher-is-better** convention, it is **proper** when

$$
\mathbb{E}_{q(\boldsymbol{x},y)}[S(p,(y,\boldsymbol{x}))]\leq
\mathbb{E}_{q(\boldsymbol{x},y)}[S(q,(y,\boldsymbol{x}))].
$$

It is **strictly proper** if equality requires $p(\cdot\mid\boldsymbol{x})=q(\cdot\mid\boldsymbol{x})$ for $q(\boldsymbol{x})$-almost every $\boldsymbol{x}$.

For the logarithmic score, assuming the expectations exist,

$$
\begin{aligned}
&\mathbb{E}_{q(\boldsymbol{x},y)}[\log q(y\mid\boldsymbol{x})-\log p_\theta(y\mid\boldsymbol{x})]\\
&\qquad=\mathbb{E}_{q(\boldsymbol{x})}[\operatorname{KL}(q(\cdot\mid\boldsymbol{x})\Vert p_\theta(\cdot\mid\boldsymbol{x}))]\geq0.
\end{aligned}
$$

Both the input and outcome are integrated over (the outcome integral becomes a sum in classification). In training, we use a sample average in place of the population expectation. This is not, by itself, a guarantee of calibrated predictions from a finite, restricted model.

Reference: Gneiting and Raftery, *Strictly Proper Scoring Rules, Prediction, and Estimation* (2007), doi:10.1198/016214506000001437.

Negating a higher-is-better score gives a **lower-is-better loss**. Thus negative log-likelihood (NLL) is the loss associated with the logarithmic score. In classification it is the categorical cross-entropy. The Brier loss, $\sum_{k=0}^{K-1}(p_k-\mathbf{1}[y=k])^2$, is another strictly proper loss for categorical probabilities.

For regression, first assume $p_\theta(y\mid\boldsymbol{x})=\mathcal{N}(\mu_\theta(\boldsymbol{x}),v_\theta(\boldsymbol{x}))$, where $v_\theta=\sigma_\theta^2>0$ is the **variance**, not the standard deviation.

### TODO — Gaussian NLL
Starting from the Gaussian density, derive the per-observation negative log-likelihood in terms of the mean $\mu$ and variance $v$. Include the normalizing constant and state how to obtain the average loss over $N$ observations.

**Your derivation:** …

An alternative is a **Laplace distribution**, $p_\theta(y\mid\boldsymbol{x})=\operatorname{Laplace}(\mu_\theta(\boldsymbol{x}),b_\theta(\boldsymbol{x}))$, with location $\mu$, positive **scale** $b$, and variance $v=2b^2$.

The network below returns $(\mu,v)$ for **both** likelihoods. Therefore use $b=\sqrt{v/2}$ in the Laplace density. Do not interpret the second network output as the Laplace scale.

For the same fixed variance at every observation, the Gaussian NLL data loss has the same mean-predictor minimizers as MSE. For the same fixed Laplace scale at every observation, the Laplace NLL data loss has the same location-predictor minimizers as MAE. Fixed but unequal variances/scales instead give weighted squared/absolute error. When the variance/scale is learned, the normalizing term and the residual weighting must also be included; the NLL is no longer simply MSE or MAE.

### TODO — Laplace NLL
Starting from the Laplace density, derive its per-observation NLL in terms of $(\mu,b)$, then substitute $b=\sqrt{v/2}$ to express the loss using the network outputs $(\mu,v)$. Include the normalizing term and average over observations.

**Your derivation:** …

### Adversarial training
The fast gradient sign method (FGSM) constructs a nearby input intended to increase a model's loss:

$$
\begin{aligned}
\boldsymbol{x}'&=\boldsymbol{x}+\epsilon\,\operatorname{sign}\!\left(\nabla_{\boldsymbol{x}}\ell(\theta,\boldsymbol{x},y)\right),\\
\|\boldsymbol{x}'-\boldsymbol{x}\|_\infty&\leq\epsilon.
\end{aligned}
$$

An FGSM step need not cause a misclassification, and the construction also applies to regression. Train on the average of the clean and perturbed losses, keeping the target unchanged. Treat the constructed adversarial input as fixed for the parameter update, using `jax.lax.stop_gradient`. With `argnums=1`, the input gradient already has the same array shape as the inputs; an extra `jnp.array` conversion is unnecessary.

Here $\epsilon=0.08$ for regression and $0.02$ for classification, in the original input units. This assumes the unchanged target remains useful within that neighborhood. For the cubic regression function, this is only approximate, especially at large $|x|$. Adversarial training is an experimental comparison, not a guarantee of improved calibration. See [Goodfellow et al.](https://arxiv.org/abs/1412.6572) and the [deep-ensembles paper](https://arxiv.org/abs/1612.01474). [Virtual adversarial training](https://arxiv.org/abs/1704.03976) is a related method that does not require the true target for its consistency term.

### Ensembles
We use separately initialized networks and average their predictive **distributions**:

$$
p(y\mid\boldsymbol{x})=\frac{1}{M}\sum_{m=1}^M p_{\theta_m}(y\mid\boldsymbol{x}).
$$

For classification, average probabilities, not logits. For regression, the exact prediction is a mixture, not generally a single Gaussian or Laplace distribution. If member $m$ has mean $\mu_m$ and variance $v_m$, its mixture moments are

$$
\begin{aligned}
\mu_{\ast}&=\frac{1}{M}\sum_m\mu_m,\\
v_{\ast}&=\frac{1}{M}\sum_m v_m+\frac{1}{M}\sum_m(\mu_m-\mu_{\ast})^2.
\end{aligned}
$$

This is the law of total variance: average within-member variance plus between-member mean variance. It applies to both likelihoods because the network always returns a variance. For Laplace components expressed instead in scales $b_m$, the first term is $\frac{1}{M}\sum_m2b_m^2$; **do not multiply the returned variances by two again**.

A Gaussian with these moments is a moment-matched approximation, not the exact mixture. A shaded mean $\pm$ one-standard-deviation band is not automatically a calibrated prediction interval with a particular coverage.

The toy classification code uses **full-batch** updates. Shuffling the entire dataset cannot change the mathematical full-batch objective; independent initialization is the source of diversity here. Mini-batch ordering would introduce an additional source of training randomness.

## Regression on toy dataset

In [ ]:
# Colab: use the runtime's existing JAX, NumPy, and Matplotlib.
# Only if an import reports a missing package, install it before importing:
# %pip install jax numpy matplotlib
# Then restart the runtime. For local Jupyter setup, see README.md.

In [ ]:
import jax
import jax.numpy as jnp
from jax import grad, jit, vmap, value_and_grad, random
from jax.scipy.special import logsumexp
from jax.example_libraries import optimizers
from functools import partial
import matplotlib.pyplot as plt
import numpy as np
import time

print("JAX:", jax.__version__, "| Devices:", jax.devices())

### Positive predictive variance
The second raw network output $\rho\in\mathbb{R}$ is transformed into

$$
v=\sigma^2=\operatorname{softplus}(\rho)+10^{-6}>0.
$$

Use the numerically stable `jax.nn.softplus`, rather than implementing `log(1 + exp(rho))` directly. This floor avoids a zero variance; it does not prevent all forms of overfitting or establish calibration. In the Laplace loss, convert this variance to $b=\sqrt{v/2}$.

In [ ]:
# A helper function to randomly initialize weights and biases
# for a dense neural network layer
def random_regression_layer_params(m, n, key, scale=1.0):
    """Fan-in-scaled normal weights and zero biases; scale is a multiplier."""
    return scale * np.sqrt(2.0 / m) * random.normal(key, (n, m)), jnp.zeros(n)

# Initialize all layers for a fully-connected neural network with sizes "sizes"
def init_regression_params(sizes, key, scale):
    keys = random.split(key, len(sizes))
    return [random_regression_layer_params(m, n, k, scale) for m, n, k in zip(sizes[:-1], sizes[1:], keys)]

def predict_regression(params, inputs):
    # per-example predictions
    activations = inputs
    for w, b in params[:-1]:
        outputs = jnp.dot(w, activations) + b
        activations = jax.nn.relu(outputs)

    final_w, final_b = params[-1]
    scores = jnp.dot(final_w, activations) + final_b

    mean, rho = jnp.split(scores, 2)

    variance = jax.nn.softplus(rho) + 1e-6
    return mean, variance

batched_predict_regression = vmap(predict_regression, in_axes=(None, 0))

def loss_mse(params, inputs, targets):
    """Optional point-prediction baseline; this does not train the variance head."""
    mean, _ = batched_predict_regression(params, inputs)
    return jnp.mean((mean - targets)**2)

def loss_mae(params, inputs, targets):
    """Optional point-prediction baseline; this does not train the variance head."""
    mean, _ = batched_predict_regression(params, inputs)
    return jnp.mean(jnp.abs(mean - targets))

def NLLloss_gaussian(params, inputs, targets):
    """TODO: Return the average Gaussian NLL, including its normalizing constant.

    inputs and targets: (N, 1). batched_predict_regression returns mean and
    variance, each (N, 1). The result must be a scalar, not a per-example array.
    """
    raise NotImplementedError("TODO: Gaussian NLL")

def NLLloss_laplacian(params, inputs, targets):
    """TODO: Return the average Laplace NLL, including its normalizing term.

    inputs and targets: (N, 1). The network returns a VARIANCE, not a scale.
    Obtain the Laplace scale using b = sqrt(variance / 2).
    """
    raise NotImplementedError("TODO: Laplace NLL")

def NLLlossAT_gaussian(params, inputs, targets, eps=0.08):
    """TODO: Average clean and FGSM-perturbed Gaussian NLLs.

    inputs and targets: (N, 1). Obtain the input gradient with
    value_and_grad(NLLloss_gaussian, argnums=1). Keep targets unchanged,
    and use stop_gradient on the constructed adversarial inputs.
    """
    raise NotImplementedError("TODO: adversarial Gaussian NLL")

def NLLlossAT_laplacian(params, inputs, targets, eps=0.08):
    """TODO: Apply the same FGSM construction using the Laplace NLL.

    inputs and targets: (N, 1). Keep targets unchanged, and average the
    clean and adversarial losses. eps is in the original input units.
    """
    raise NotImplementedError("TODO: adversarial Laplace NLL")

@partial(jit, static_argnames=("loss_fn", "opt_update", "get_params"))
def update_regression(step, params, x, y, opt_state, loss_fn, opt_update, get_params):
    """One full-batch update; step is the zero-based optimizer update count."""
    value, grads = value_and_grad(loss_fn)(params, x, y)
    opt_state = opt_update(step, grads, opt_state)
    return get_params(opt_state), opt_state, value

def data_set(points=20, xrange=(-4, 4), std=3., seed=173):
    rng = np.random.RandomState(seed)
    xx = rng.uniform(*xrange, size=(points, 1))
    yy = xx**3 + rng.normal(0, std, size=(points, 1))
    return jnp.asarray(xx, dtype=jnp.float32), jnp.asarray(yy, dtype=jnp.float32)

In [ ]:
# Regression settings. Change these, then restart and run all cells.
layer_sizes = [1, 10, 6, 30, 2]
num_epochs = 600
step_size = 4e-2
M = 6
likelihood = "gaussian"       # "gaussian" or "laplacian"
adversarial_training = False  # compare False and True with the same seeds
param_scale = 1.0             # multiplier on fan-in-scaled normal initialization
# Fan-in scaling and lr=0.04 improve training stability without changing layer widths.

assert M >= 1 and num_epochs >= 1
assert likelihood in {"gaussian", "laplacian"}
loss_functions = {
    ("gaussian", False): NLLloss_gaussian,
    ("gaussian", True): NLLlossAT_gaussian,
    ("laplacian", False): NLLloss_laplacian,
    ("laplacian", True): NLLlossAT_laplacian,
}
regression_loss = loss_functions[(likelihood, adversarial_training)]
xx, yy = data_set(points=40, xrange=(-4, 4), std=3., seed=173)
x = jnp.linspace(-6, 6, 100).reshape(-1, 1)
y = x**3  # noise-free conditional mean, not a noisy target sample

params_all, mean_all, var_all = [], [], []
regression_initial_losses, regression_final_losses = [], []
reg_opt_init, reg_opt_update, reg_get_params = optimizers.adam(step_size)
start_time = time.perf_counter()
for i in range(M):
    params = init_regression_params(layer_sizes, random.PRNGKey(i * 10), param_scale)
    opt_state = reg_opt_init(params)
    initial_loss = float(regression_loss(params, xx, yy))
    for epoch in range(num_epochs):
        params, opt_state, train_loss = update_regression(
            epoch, params, xx, yy, opt_state,
            regression_loss, reg_opt_update, reg_get_params)
    final_loss = float(regression_loss(params, xx, yy))
    assert np.isfinite(final_loss), "Non-finite regression loss; inspect the training settings."
    params_all.append(params)
    regression_initial_losses.append(initial_loss)
    regression_final_losses.append(final_loss)
    print(f"Member {i}: initial NLL {initial_loss:.4f}; final NLL {final_loss:.4f}")
    mean, variance = batched_predict_regression(params, x)
    mean_all.append(mean)
    var_all.append(variance)

mean_all, var_all = np.asarray(mean_all), np.asarray(var_all)
mean_ = mean_all.mean(axis=0)
# TODO: Use the law of total variance for this uniformly weighted mixture.
# The second network output is a variance for BOTH likelihood choices.
raise NotImplementedError("TODO: mixture variance")
std_ = np.sqrt(var_)
assert mean_.shape == var_.shape == (len(x), 1)
assert np.isfinite(mean_).all() and np.isfinite(var_).all() and (var_ > 0).all()
# Retain regression results before the classification section reuses variable names.
regression_models = params_all
regression_seconds = time.perf_counter() - start_time
print(f"Trained {M} regression members in {regression_seconds:.2f} s ({likelihood}, AT={adversarial_training}).")

In [ ]:
# visualize ensemble output
fig = plt.figure(constrained_layout=True,
                 figsize=(15, 7),
                 dpi=100)
fig.suptitle('Predictive mean ± 1 standard deviation (not a guaranteed-coverage interval)',
             fontsize='xx-large')
subfigs = fig.subfigures(1, 2, wspace=0.07)
axs_left = subfigs[0].subplots(int(np.ceil(M / 2)), 2, squeeze=False)
for i,ax in zip(range(M),axs_left.flat):
    ax.plot(x, y, "b-")
    ax.plot(xx,yy,"or")
    ax.plot(x, mean_all[i])
    ax.fill_between(x.reshape(-1,), (mean_all[i]-np.sqrt(var_all[i])).reshape(-1,), (mean_all[i]+np.sqrt(var_all[i])).reshape(-1,),alpha=0.5)
    ax.grid()
    ax.set_title("MLP " + str(i))
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_ylim((-300,300))

for ax in axs_left.flat[M:]:
    ax.set_visible(False)

axs_right = subfigs[1].subplots(1, 1)
axs_right.plot(x, y, "b-", label="conditional mean: $y=x^3$")
axs_right.plot(xx,yy,"or", label="data points")
axs_right.plot(x, mean_, label="mean prediction", color="grey")
axs_right.fill_between(x.reshape(-1,), (mean_-std_).reshape(-1,), (mean_+std_).reshape(-1,),alpha=0.5)
axs_right.grid()
axs_right.set_title("Uncertainty of an Ensemble of "+str(M)+" MLPs")
axs_right.set_xlabel("x")
axs_right.set_ylabel("y")
axs_right.set_ylim((-300,300))
axs_right.legend()
plt.show()

### Discussion — regression
Compare the two likelihoods and clean versus adversarial training while keeping the data, initialization seeds, architecture, and update budget fixed. Compare $M=1$ with $M=6$; the first $M$ seeds are shared so that adding members does not change existing initializations.

1. How do adversarial training and ensembling change the predicted mean and variance? Separate average within-member variance from between-member mean disagreement.
2. How do the results change when $M$ increases? Contrast interpolation ($x\in[-4,4]$) with extrapolation. Do not assume that more members always improve calibration or make uncertainty larger.

A shaded one-standard-deviation band alone is insufficient evidence of calibration. The optional MSE/MAE helpers are point-prediction baselines; they do not learn a predictive variance and are not selectable uncertainty models in the training cell.

## Classification on [toy dataset] :
[toy dataset]: https://cs231n.github.io/neural-networks-case-study/

For one-hot targets $\boldsymbol{t}_n$ and class probabilities $p_{nk}$,

$$
\begin{aligned}
p_\theta(y_n\mid\boldsymbol{x}_n)&=\prod_{k=0}^{K-1}p_{nk}^{t_{nk}},\\
-\log p_\theta(y_n\mid\boldsymbol{x}_n)&=-\sum_{k=0}^{K-1}t_{nk}\log p_{nk}=-\log p_{n,y_n}.
\end{aligned}
$$

The average NLL is $-\frac{1}{N}\sum_n\sum_k t_{nk}\log p_{nk}$: **sum over classes, average over examples**. Use `log_softmax` or `logsumexp` rather than taking the logarithm of possibly underflowed probabilities.

To preserve the original regularization choice, the code uses $\lambda\|\theta\|_2$ over all parameters. This is an **L2-norm penalty**, not the more usual squared-L2 penalty $\lambda\|\theta\|_2^2/2$, and not decoupled AdamW weight decay. The variable `wd` is retained for compatibility but denotes the coefficient of this explicitly defined penalty.

In [ ]:
def one_hot(x, k, dtype=jnp.float32):
    """One-hot encode integer class labels."""
    return jnp.asarray(x[:, None] == jnp.arange(k), dtype)

def generate_datasets(N, K, noise, seed=0):
    """N points per class in a two-dimensional spiral; local, reproducible RNG."""
    rng = np.random.RandomState(seed)
    X = np.zeros((N * K, 2), dtype=np.float32)
    y = np.zeros(N * K, dtype=np.int32)
    for j in range(K):
        ix = slice(N * j, N * (j + 1))
        r = np.linspace(0.0, 1.0, N)
        t = np.linspace(j * 8, (j + 1) * 8, N) + rng.randn(N) * noise * (r + 1.0)
        X[ix] = np.c_[r * np.sin(t), r * np.cos(t)]
        y[ix] = j
    X, y = jnp.asarray(X), jnp.asarray(y)
    return X, y, one_hot(y, K)

In [ ]:
N = 1000  # points per class
D = 2
K = 3
X, y, y_onehot = generate_datasets(N, K, noise=0.3, seed=0)
# Independent noise realization on the same radial grid; never used for training.
X_test, y_test, y_test_onehot = generate_datasets(N, K, noise=0.3, seed=1)
plt.figure()
plt.scatter(X[:, 0], X[:, 1], c=y, s=15, cmap=plt.cm.Spectral)
plt.title("Training data")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()
print("Training:", X.shape, y.shape, "| Held-out:", X_test.shape, y_test.shape)

In [ ]:
# A helper function to randomly initialize weights and biases
# for a dense neural network layer
def random_layer_params(m, n, key, scale=1e-1):
    w_key, b_key = random.split(key)
    return scale * random.normal(w_key, (n, m)), scale * random.normal(b_key, (n,))

# Initialize all layers for a fully-connected neural network with sizes "sizes"
def init_network_params(sizes, key):
    keys = random.split(key, len(sizes))
    return [random_layer_params(m, n, k) for m, n, k in zip(sizes[:-1], sizes[1:], keys)]

@jit
def predict(params, image):
    # per-example predictions
    activations = image
    for w, b in params[:-1]:
        outputs = jnp.dot(w, activations) + b
        activations = jax.nn.relu(outputs)

    final_w, final_b = params[-1]
    scores = jnp.dot(final_w, activations) + final_b
    probs = jax.nn.softmax(scores)
    return scores, probs

batched_predict = vmap(predict, in_axes=(None, 0))

def accuracy(params, inputs, targets):
    target_class = jnp.argmax(targets, axis=1)
    scores, _ = batched_predict(params, inputs)
    predicted_class = jnp.argmax(scores, axis=1)
    return jnp.mean(predicted_class == target_class)

def NLLloss_categorical(params, inputs, targets):
    """Mean categorical NLL: inputs (N, D), one-hot targets (N, K)."""
    scores, _ = batched_predict(params, inputs)
    log_probs = jax.nn.log_softmax(scores, axis=-1)
    return -jnp.mean(jnp.sum(log_probs * targets, axis=-1))

def l2_norm_regularization(params):
    """Original L2-norm penalty (not squared L2 or AdamW)."""
    return optimizers.l2_norm(params)

def loss(params, inputs, targets, wd):
    return NLLloss_categorical(params, inputs, targets) + wd * l2_norm_regularization(params)

def NLLlossAT_categorical(params, inputs, targets, eps=0.02):
    """TODO: Average clean and FGSM-perturbed categorical NLLs.

    inputs: (N, D), here D=2. One-hot targets: (N, K), here K=3.
    Differentiate the NLL with respect to inputs (argnums=1), preserve
    the targets, and use stop_gradient on the adversarial inputs.
    """
    raise NotImplementedError("TODO: adversarial categorical NLL")

def lossAT(params, inputs, targets, wd):
    return NLLlossAT_categorical(params, inputs, targets) + wd * l2_norm_regularization(params)

@partial(jit, static_argnames=("adversarial", "opt_update", "get_params"))
def update(step, params, x, y, opt_state, wd, adversarial, opt_update, get_params):
    loss_fn = lossAT if adversarial else loss
    value, grads = value_and_grad(loss_fn)(params, x, y, wd)
    opt_state = opt_update(step, grads, opt_state)
    return get_params(opt_state), opt_state, value

@jit
def ensemble_log_probabilities(params_all, inputs):
    log_probs = jnp.stack([jax.nn.log_softmax(batched_predict(params, inputs)[0], axis=-1)
                           for params in params_all])
    return logsumexp(log_probs, axis=0) - jnp.log(len(params_all))

def logging(params_all, X, y, log_acc_train, log_nll_train):
    log_probs = ensemble_log_probabilities(params_all, X)
    log_acc_train.append(float(jnp.mean(jnp.argmax(log_probs, axis=-1) == y)))
    log_nll_train.append(float(-jnp.mean(log_probs[jnp.arange(len(y)), y])))

In [ ]:
# Classification settings. Use the same settings/seeds for the AT comparison.
layer_sizes = [D, 150, 150, K]
num_epochs = 300
step_size = 0.1
wd = 1e-3  # coefficient of the L2-NORM penalty defined above
M = 4
adversarial_training = False

assert M >= 1 and num_epochs >= 1
opt_init, opt_update, get_params = optimizers.adam(step_size)
params_all = [init_network_params(layer_sizes, random.PRNGKey(i)) for i in range(M)]
opt_state_all = [opt_init(params) for params in params_all]
log_acc, log_nll = [], []
start_time = time.perf_counter()
for epoch in range(num_epochs):
    for i in range(M):
        # One full-batch update per member: epoch is that member's update count.
        params_all[i], opt_state_all[i], train_loss = update(
            epoch, params_all[i], X, y_onehot, opt_state_all[i], wd,
            adversarial_training, opt_update, get_params)
    logging(params_all, X, y, log_acc, log_nll)
    if epoch == 0 or (epoch + 1) % 50 == 0 or epoch + 1 == num_epochs:
        print(f"Epoch {epoch+1}: train accuracy {log_acc[-1]:.4f}; train NLL {log_nll[-1]:.4f}")
assert np.isfinite(log_nll).all(), "Non-finite classification NLL."
classification_seconds = time.perf_counter() - start_time
heldout_acc, heldout_nll = [], []
logging(params_all, X_test, y_test, heldout_acc, heldout_nll)
print(f"Held-out accuracy {heldout_acc[-1]:.4f}; held-out NLL {heldout_nll[-1]:.4f}")
print(f"Training time: {classification_seconds:.2f} s; parameters: {sum(p.size for params in params_all for p in jax.tree_util.tree_leaves(params)):,}")

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(10,5))
fig.tight_layout()
axes[0].plot(log_acc)
axes[1].plot(log_nll)
axes[0].title.set_text('Train Acc')
axes[1].title.set_text('Train NLL')

### Predictive entropy and ensemble disagreement
For a categorical distribution $\boldsymbol{p}$, Shannon entropy is

$$
H(\boldsymbol{p})=-\sum_{k=0}^{K-1}p_k\log p_k,
$$

with $0\log0=0$. Natural logarithms give entropy in **nats**, in the range $[0,\log K]$.

Let $\boldsymbol{p}^{(m)}$ be the prediction of member $m$, and $\bar{\boldsymbol{p}}=M^{-1}\sum_m\boldsymbol{p}^{(m)}$. Following the terminology of [Malinin et al., *Ensemble Distribution Distillation*](https://arxiv.org/abs/1905.00076), define

$$
\begin{aligned}
U_{\mathrm{total}}&=H(\bar{\boldsymbol{p}}),\\
U_{\mathrm{data}}&=\frac{1}{M}\sum_mH(\boldsymbol{p}^{(m)}),\\
U_{\mathrm{knowledge}}&=U_{\mathrm{total}}-U_{\mathrm{data}}.
\end{aligned}
$$

The last expression is the equally weighted generalized Jensen–Shannon divergence, and equals the mutual information between a uniformly sampled **member index** and a label sampled from that member. It is nonnegative, bounded by $\min(\log M,\log K)$, and is zero when $M=1$ or all members agree.

These are exact quantities **within this fitted ensemble**. Interpreting average entropy as irreducible data/aleatoric uncertainty, and disagreement as all epistemic uncertainty, requires additional modeling assumptions. Independently trained networks are not automatically posterior samples. Members can agree confidently and still be wrong, particularly outside the training support.

The visualizations use the same $[0,\log K]$ color scale for all uncertainty maps, including across ensemble sizes. Predictive entropy is shown separately in addition to the original data/disagreement maps.

In [ ]:
def entropy(preds):
    """Categorical entropy in nats along the final axis; exactly 0*log(0)=0."""
    preds = np.asarray(preds, dtype=np.float64)
    safe = np.where(preds > 0, preds, 1.0)
    return -np.sum(preds * np.log(safe), axis=-1)

def js_terms(distributions):
    """Input (M, N, K); return total and average member entropy, each (N,)."""
    distributions = np.asarray(distributions)
    return entropy(distributions.mean(axis=0)), entropy(distributions).mean(axis=0)

def visualize_predictions(X, y, params_list, min=-2.0, max=2.0, res=200, num_nets=None):
    xs = np.linspace(min, max, res)
    ys = np.linspace(min, max, res)
    xy = np.asarray([(_x, _y) for _x in xs for _y in ys], dtype=np.float32)
    predictions = np.asarray([batched_predict(params, xy)[1] for params in params_list])
    predictions_ensemble = predictions.mean(axis=0)
    total, data_uncertainty = js_terms(predictions)
    knowledge = np.maximum(total - data_uncertainty, 0.0)  # roundoff only
    K_plot = predictions.shape[-1]
    levels = np.linspace(0, np.log(K_plot), 51)
    shape = (len(xs), len(ys))
    labels = np.argmax(predictions_ensemble, axis=-1).reshape(shape).T
    fig, axes = plt.subplots(2, 2, figsize=(10, 9), constrained_layout=True)
    axes = axes.flatten()
    class_norm = plt.matplotlib.colors.BoundaryNorm(np.arange(K_plot + 1) - 0.5, plt.cm.Spectral.N)
    axes[0].scatter(X[:, 0], X[:, 1], c=y, s=10, cmap=plt.cm.Spectral, norm=class_norm)
    axes[1].contourf(xs, ys, labels, levels=np.arange(K_plot + 1) - 0.5, cmap=plt.cm.Spectral, norm=class_norm)
    for ax, values, title in [(axes[2], data_uncertainty, "Average member entropy (data proxy)"),
                              (axes[3], knowledge, "Member disagreement (knowledge proxy)")]:
        contour = ax.contourf(xs, ys, values.reshape(shape).T, levels=levels, cmap="magma")
        fig.colorbar(contour, ax=ax, label="nats")
        ax.set_title(title)
    axes[0].set_title("Training data")
    axes[1].set_title("Ensemble predicted class")
    for ax in axes:
        ax.set(xlim=(min, max), ylim=(min, max), xlabel="x1", ylabel="x2", aspect="equal")
    plt.show()
    fig, ax = plt.subplots(figsize=(5, 4), constrained_layout=True)
    contour = ax.contourf(xs, ys, total.reshape(shape).T, levels=levels, cmap="magma")
    fig.colorbar(contour, ax=ax, label="nats")
    ax.set(title="Total predictive entropy", xlabel="x1", ylabel="x2", aspect="equal")
    plt.show()

In [ ]:
visualize_predictions(X, y, params_all, num_nets=M)

### Discussion — classification
1. Identify regions of low predictive entropy outside the training support (the unit disk). Explain why these may indicate **unjustified confidence**. Without a specified ground-truth label distribution there, do not claim that the predictions are known to be wrong.
2. Where is total predictive entropy high, and which of average member entropy and disagreement explains it? Where do members disagree most? Support the answer with the plotted values rather than color intensity alone.
3. Repeat with `adversarial_training=True`, using the same data, model seeds and update budget. Compare the uncertainty maps and held-out accuracy/NLL. Does adversarial training help every metric? Explain plausible causes without asserting a general improvement from one run.
4. Report your own approximate completion time. This is a student self-report, not a quantity inferred from training runtime.

The held-out sample checks performance on a fresh noise realization of the same synthetic data construction. It is not an out-of-distribution benchmark or a complete calibration assessment.